# Simulacion Roland Garros

In [1]:
import random
import pickle
import pandas as pd

In [2]:
from features import (forma_reciente, winrate, headtohead, experiencia, 
                      get_ranking, get_elo)

In [3]:
# WTA limpio como fuente de consulta para generar los features del partido a simular
wta=pd.read_csv('wta_limpio.csv', parse_dates=['Date'])


In [4]:
# Cargar el modelo de ML
with open ('gbx_v3.model', 'rb') as archivo_entrada:
    modeloML = pickle.load(archivo_entrada)
print(modeloML)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['rank_diff', 'wins2meses_p1',
                                                   'wins2meses_p2',
                                                   'ratio_superficie_p1',
                                                   'ratio_superficie_p2', 'h2h',
                                                   'ratio_ronda_p1',
                                                   'ratio_ronda_p2',
                                                   'experiencia_p1',
                                                   'experiencia_p2',
                                                   'is_new_p1', 'is_new_p2',
                                                   'elo_p1', 'elo_p2',
                                                   'elo_diff', 'elo_global_p1',
                                                   'elo_global_p2',
                

┌─────────────────────────────────────────┐
│           SIMULACIÓN MONTE CARLO        │  ← repite 10.000 veces
│                                         │
│   Cuartos → Semis → Final               │
│   para cada partido llama a...          │
│                                         │
│   ┌─────────────────────────────────┐   │
│   │         MODELO ML               │   │  ← entrenado con datos históricos
│   │  input: jugadora A vs B         │   │
│   │  output: probabilidad 0..1      │   │
│   └─────────────────────────────────┘   │
│                                         │
└─────────────────────────────────────────┘
         ↓ resultado final
   {"Swiatek": 38%, "Sabalenka": 22%, ...}

In [18]:
# cargar el cuadro generado del archivo de texto
# el archivo contiene código Python válido que crea la variable cuadro
with open("cuadro_python_list.txt", "r", encoding="utf-8") as f:
    exec(f.read())

print(cuadro[:10])

['Sabalenka A.', 'Siniakova K.', 'Masarova R.', 'Ruzic A.', 'Krueger A.', 'Osorio C.', 'Yastremska D.', 'Bucsa C.', 'Wang X.', 'Townsend T.']


In [ ]:
# La simulación tarda muchísimo si tiene que calcular cada vez (10.000) todas las features. 
# Como las jugadoras ya las sabemos, calculamos las features antes, las almacenamos y luego llamamos ahí 
# en vez de a las funciones

In [ ]:
def calculo_features_jugadoras_torneo (df, cuadro):
    fecha = pd.to_datetime('2026-05-05')
    superficie = 'Clay'
    rondas = ['1st Round', '2nd Round', '3rd Round', '4th Round', 
              'Quarterfinals', 'Semifinals', 'The Final']
    
    cache_features = {}
    for jugadora in cuadro:

        # Calcular los valores
        wins2meses_val = forma_reciente(df, jugadora, fecha)
        ratio_superficie_val = winrate(df, jugadora, fecha, superficie=superficie)
        experiencia_val = experiencia(df, jugadora, fecha)
        ranking_val = get_ranking(df, jugadora, fecha)
        elo_global = get_elo (df, jugadora, fecha)
        elo_superficie = get_elo (df,jugadora,fecha,superficie)
        
        # Calcular winrate por ronda
        winrate_por_ronda = {}
        for ronda in rondas:
            winrate_por_ronda[ronda] = winrate(df, jugadora, fecha, ronda=ronda)
    
        cache_features[jugadora] = {
            'wins2meses': wins2meses_val,
            'ratio_superficie': ratio_superficie_val,
            'experiencia': experiencia_val,
            'ranking': ranking_val,
            'winrate_por_ronda': winrate_por_ronda,
            'elo_global':   elo_global,
            'elo_superficie': elo_superficie
        }

    # Y una caché para los h2h entre cada par
    cache_h2h = {}
    for i, p1 in enumerate(cuadro):
        for p2 in cuadro[i+1:]:
            cache_h2h[(p1, p2)] = headtohead(df, p1, p2, fecha)
            cache_h2h[(p2, p1)] = 1 - cache_h2h[(p1, p2)]
    
    return cache_features, cache_h2h

def construir_features_desde_cache (cache_features, cache_h2h, p1, p2, superficie, ronda):
    f1 = cache_features[p1]
    f2 = cache_features[p2]

    row = {
        'surface':             superficie,
        'round':               ronda,
        'rank_diff':           f1['ranking'] - f2['ranking'],
        'wins2meses_p1':       f1['wins2meses'],
        'wins2meses_p2':       f2['wins2meses'],
        'ratio_superficie_p1': f1['ratio_superficie'],
        'ratio_superficie_p2': f2['ratio_superficie'],
        'h2h':                 cache_h2h.get((p1, p2), 0.5),
        'ratio_ronda_p1':      f1['winrate_por_ronda'][ronda],
        'ratio_ronda_p2':      f2['winrate_por_ronda'][ronda],
        'experiencia_p1':      f1['experiencia'],
        'experiencia_p2':      f2['experiencia'],
        'tournament_type':     'GS',
        'elo_p1':          f1['elo_superficie'],
        'elo_p2':          f2['elo_superficie'],
        'elo_diff':        f1['elo_superficie'] - f2['elo_superficie'],
        'elo_global_p1':   f1['elo_global'],
        'elo_global_p2':   f2['elo_global'],
        'elo_global_diff': f1['elo_global'] - f2['elo_global'],
        'is_new_p1': int(f1['experiencia'] < 10),
        'is_new_p2': int(f2['experiencia'] < 10),

    }
    return pd.DataFrame([row])

In [19]:
cache_features, cache_h2h = calculo_features_jugadoras_torneo (wta, cuadro)

In [ ]:
# X = construir_features_desde_cache(cache_features, cache_h2h, cuadro[0], cuadro[1], 'Clay', '1st Round', pd.to_datetime('2026-05-05'))
# print(X.columns.tolist())           # columnas que genera la simulación
# print(modeloML.feature_names_in_)  # columnas que espera el modelo

['surface', 'round', 'rank_diff', 'wins2meses_p1', 'wins2meses_p2', 'ratio_superficie_p1', 'ratio_superficie_p2', 'h2h', 'ratio_ronda_p1', 'ratio_ronda_p2', 'experiencia_p1', 'experiencia_p2', 'tournament_type', 'elo_p1', 'elo_p2', 'elo_diff', 'elo_global_p1', 'elo_global_p2', 'elo_global_diff', 'is_new_p1', 'is_new_p2']
['surface' 'round' 'tournament_type' 'rank_diff' 'wins2meses_p1'
 'wins2meses_p2' 'ratio_superficie_p1' 'ratio_superficie_p2' 'h2h'
 'ratio_ronda_p1' 'ratio_ronda_p2' 'experiencia_p1' 'experiencia_p2'
 'is_new_p1' 'is_new_p2' 'elo_p1' 'elo_p2' 'elo_diff' 'elo_global_p1'
 'elo_global_p2' 'elo_global_diff']


In [ ]:
# FUNCIONES PARA LA SIMULACIÓN DEL TORNEO

def simular_partido(prob_a):
  # lanzamos un dado cargado
  return random.random() < prob_a

def simular_torneo(cache_features, cache_h2h, cuadro, modelo):
    fecha = pd.to_datetime('2026-05-05')
    superficie = 'Clay'
    rondas = ['1st Round', '2nd Round', '3rd Round', '4th Round', 
              'Quarterfinals', 'Semifinals', 'The Final']
    
    jugadoras = cuadro.copy()
    indice_ronda = 0 

    while len(jugadoras) > 1:
        siguiente_ronda = []
        ronda_actual = rondas[indice_ronda]  # Nombre de la ronda actual
   
        # Emparejar jugadoras
        for i in range(0, len(jugadoras), 2):
            a, b = jugadoras[i], jugadoras[i+1]
            X = construir_features_desde_cache (cache_features, cache_h2h, a, b, superficie, ronda_actual)
            prob_a = modelo.predict_proba(X)[0][1]  # Clase 1: probabilidad de que gane p1(a)
            
            ganadora = a if simular_partido(prob_a) else b
            siguiente_ronda.append(ganadora)
        
        # Actualizar para la siguiente ronda
        jugadoras = siguiente_ronda
        indice_ronda += 1  
        

    return jugadoras[0]

In [ ]:

# SIMULACIÓN MONTE CARLO
def simulacion_montecarlo(cache_features, cacheh2h, cuadro, modelo, n_simulaciones=10000):

    victorias = {}  # Diccionario vacío
    
    for _ in range(n_simulaciones):
        campeona = simular_torneo(cache_features, cacheh2h, cuadro, modelo)

        if campeona in victorias:
            victorias[campeona] += 1
        else:
            victorias[campeona] = 1
    
    # Calcular probabilidades
    probabilidades = {}
    for jugadora, wins in victorias.items():
        probabilidades[jugadora] = wins / n_simulaciones
    
    return probabilidades, victorias

# # Cuadro ficticio prueba
# def crear_cuadro_top16():
#     return [
#         "Swiatek I.", "Sabalenka A.", "Gauff C.", "Rybakina E.",
#         "Pegula J.", "Vondrousova", "Jabeur O.", "Zheng",
#         "Sakkari M.", "J. Ostapenko", "Badosa P.", "D. Kasatkina",
#         "Keys M.", "Azarenka V.", "Svitolina E.", "Navarro E."
#     ]



In [12]:
# Cuadro ficticio prueba
def crear_cuadro_top16():
    return [
        "Swiatek I.", "Sabalenka A.", "Gauff C.", "Rybakina E.",
        "Pegula J.", "Vondrousova", "Jabeur O.", "Zheng",
        "Sakkari M.", "J. Ostapenko", "Badosa P.", "D. Kasatkina",
        "Keys M.", "Azarenka V.", "Svitolina E.", "Navarro E."
    ]

In [13]:
cuadro16 = crear_cuadro_top16()

In [ ]:
import time
inicio = time.time()
probabilidades, victorias = simulacion_montecarlo(cache_features, cache_h2h, cuadro, modeloML, 100)
fin = time.time()
tiempo_100 = fin - inicio

# Mostrar resultados
print("Resultados después de 10,000 simulaciones:")
print("-" * 40)
for jugadora, prob in sorted(probabilidades.items(), key=lambda x: x[1], reverse=True):
    print(f"{jugadora}: {prob:.2%} ({victorias[jugadora]} títulos)")
print(f"Tiempo de simulación: {tiempo_100:.2f} segundos")

Resultados después de 10,000 simulaciones:
----------------------------------------
Swiatek I.: 24.00% (24 títulos)
Sabalenka A.: 20.00% (20 títulos)
Gauff C.: 11.00% (11 títulos)
Rybakina E.: 11.00% (11 títulos)
Muchova K.: 7.00% (7 títulos)
Andreeva M.: 6.00% (6 títulos)
Anisimova A.: 6.00% (6 títulos)
Pegula J.: 4.00% (4 títulos)
Svitolina E.: 4.00% (4 títulos)
Shnaider D.: 1.00% (1 títulos)
Vondrousova M.: 1.00% (1 títulos)
Li A.: 1.00% (1 títulos)
Mboko V.: 1.00% (1 títulos)
Bencic B.: 1.00% (1 títulos)
Salkova D.: 1.00% (1 títulos)
Kudermetova V.: 1.00% (1 títulos)
Tiempo de simulación: 46.33 segundos


In [ ]:
jugadora_test = cuadro[75]
print(f"Jugadora: {jugadora_test}")
X_test = construir_features_desde_cache(cache_features, cache_h2h, 'Swiatek I.', jugadora_test, 'Clay', '1st Round')
print(modeloML.predict_proba(X_test))

Jugadora: Valentova T.
[[0.0856216 0.9143784]]


In [ ]:
jugadora_test = cuadro[48]
print(f"Jugadora: {jugadora_test}")
X_test = construir_features_desde_cache(cache_features, cache_h2h, 'Swiatek I.', jugadora_test, 'Clay', '1st Round')
print(modeloML.predict_proba(X_test))

Jugadora: Jovic I.
[[0.1656447 0.8343553]]


In [14]:
# ¿La caché tiene datos diferentes para cada jugadora?
print(cache_features['I. Swiatek']['elo_global'])
print(cache_features['T. Valentova']['elo_global'])
print(cache_features['I. Jovic']['elo_global'])

1500
1500
1500


In [16]:
# ¿Cómo aparece Swiatek en wta?
wta[wta['Player_1'].str.contains('Swiatek')]['Player_1'].unique()

array(['Swiatek I.'], dtype=object)

In [39]:
import numpy as np
# Input completamente aleatorio
X_random = pd.DataFrame([{
    'surface': 'Clay', 'round': '1st Round', 'tournament_type': 'GS',
    'rank_diff': 999, 'wins2meses_p1': 0, 'wins2meses_p2': 0,
    'ratio_superficie_p1': 0, 'ratio_superficie_p2': 0,
    'h2h': 0, 'ratio_ronda_p1': 0, 'ratio_ronda_p2': 0,
    'experiencia_p1': 0, 'experiencia_p2': 0
}])
print(modeloML.predict_proba(X_random))

[[0.680313 0.319687]]
